# Validation Base Template — DuckDB Workflow

**Template ID:** VALIDATION-BASE  
**Workflow layer:** `raw → staging → curated` (before `output`)  
**Primary mode:** Notebook + SQL via DuckDB Python API

## Purpose

Notebook-first validation template for DuckDB ETL workflows. Run row-count reconciliation, field-level gates, referential integrity, domain checks, aggregate reconciliation, and spatial validity checks across pipeline layers.

Follows the layered workflow convention:

```text
source → raw → staging → curated → output
```

## Convention

**Most validation queries return zero rows when passing.** Summary checks (row counts, aggregates) store metrics in the validation summary table; violation checks list failing records or counts.

## How to use

1. Copy this notebook into your project (or run it from this repo).
2. Update **Project configuration** with table names, column lists, and check parameters.
3. Ensure `raw`, `staging`, and `curated` tables exist (run `notebooks/01_etl_base.ipynb` first, or set `BOOTSTRAP_EXAMPLE_PIPELINE = True`).
4. Run all cells top to bottom.
5. Review the **Validation summary table** — all checks should show `PASS` before export.
6. Record **Final validation notes** at the end.

**Example tables:** `raw.raw_orders`, `staging.stg_orders`, `curated.fct_orders`  
**Target users:** analysts, data engineers, GIS users, SQL users, Python users.

## 2. Project configuration

Edit the variables below for your pipeline. Leave lists empty (`[]`), dicts empty (`{}`), or `GEOMETRY_COLUMN = None` when a section does not apply.

| Variable | Purpose |
|----------|----------|
| `RAW_TABLE_NAME` | Source-aligned layer to reconcile against |
| `STAGING_TABLE_NAME` | Cleaned layer — primary target for field checks |
| `CURATED_TABLE_NAME` | Business-ready layer — final gate before `output` |
| `PRIMARY_KEY_COLUMNS` | Composite or single-column keys for uniqueness |
| `REQUIRED_COLUMNS` | Columns that must not be NULL |
| `DATE_COLUMNS` | Date/timestamp columns for range checks |
| `NUMERIC_COLUMNS` | Numeric columns for value-range checks |
| `CATEGORY_COLUMNS` | Map of column → allowed values |
| `GEOMETRY_COLUMN` | Geometry column for spatial checks; `None` to skip |

The default example validates the TPC-H `lineitem` orders model from `notebooks/01_etl_base.ipynb`.

In [ ]:
from pathlib import Path

# --- paths ---
PROJECT_ROOT = Path("..").resolve()  # repo root when notebook lives in notebooks/
DB_PATH = PROJECT_ROOT / "work.duckdb"
VALIDATION_OUTPUT_DIR = PROJECT_ROOT / "data" / "output" / "validation"

# --- tables (schema.table) ---
RAW_TABLE_NAME = "raw.raw_orders"
STAGING_TABLE_NAME = "staging.stg_orders"
CURATED_TABLE_NAME = "curated.fct_orders"

# --- primary table for field-level checks (usually staging or curated) ---
PRIMARY_TABLE_NAME = STAGING_TABLE_NAME

# --- validation column lists ---
PRIMARY_KEY_COLUMNS = ["order_id", "line_number"]  # composite grain for lineitem example
REQUIRED_COLUMNS = ["order_id", "order_date", "amount"]
DATE_COLUMNS = ["order_date"]
NUMERIC_COLUMNS = ["amount", "quantity"]
CATEGORY_COLUMNS = {
    "order_status": ["open", "fulfilled", "returned"],
    # "ship_mode": ["AIR", "RAIL", "SHIP", "TRUCK", "REG AIR"],  # uncomment to enforce
}
GEOMETRY_COLUMN = None  # e.g. "geom" for spatial layers; None to skip spatial checks

# --- date range bounds ---
MIN_EXPECTED_DATE = "1990-01-01"
MAX_EXPECTED_DATE = "2100-12-31"
ALLOW_FUTURE_DATES = False

# --- numeric range bounds (per column) ---
NUMERIC_BOUNDS = {
    "amount": {"min": 0, "max": 1_000_000},
    "quantity": {"min": 0, "max": 10_000},
}

# --- referential integrity (optional) ---
# Each entry: child_table, child_key, parent_table, parent_key, check_name
FOREIGN_KEY_CHECKS = [
    {
        "check_name": "supplier_id → dim_suppliers",
        "child_table": STAGING_TABLE_NAME,
        "child_key": "supplier_id",
        "parent_table": "curated.dim_suppliers",
        "parent_key": "supplier_id",
    },
]

# --- aggregate reconciliation ---
AGGREGATE_COLUMN = "amount"  # numeric measure to reconcile across layers
AGGREGATE_TOLERANCE = 0.01  # allowed absolute delta between layers
STAGING_ELIGIBLE_FILTER = "order_status <> 'returned' AND amount IS NOT NULL AND amount > 0"

# --- row count reconciliation ---
# Expect counts to decrease (or stay equal) raw → staging → curated after filters
EXPECT_MONOTONIC_ROW_COUNTS = True

# --- bootstrap (runnable demo without prior ETL notebook) ---
BOOTSTRAP_EXAMPLE_PIPELINE = True  # set False when tables already exist
RAW_INPUT_PATH = "https://shell.duckdb.org/data/tpch/0_01/parquet/lineitem.parquet"
RAW_ROW_LIMIT = 50_000

# --- spatial table (when GEOMETRY_COLUMN is set) ---
SPATIAL_TABLE_NAME = CURATED_TABLE_NAME  # table to run spatial validity on
RUN_SPATIAL_CHECKS = GEOMETRY_COLUMN is not None

# --- export ---
EXPORT_VALIDATION_RESULTS = True
VIOLATION_SAMPLE_LIMIT = 100

## 3. Imports

In [ ]:
import uuid
from datetime import datetime, timezone

import duckdb
import pandas as pd
from IPython.display import display

## 4. Connect to DuckDB

Opens (or creates) a persistent database and ensures workflow schemas exist.

In [ ]:
con = duckdb.connect(str(DB_PATH))

for schema in ("raw", "staging", "curated"):
    con.execute(f'CREATE SCHEMA IF NOT EXISTS "{schema}";')

VALIDATION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_ID = f"run_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"
validation_results: list[dict] = []

print(f"Connected to: {DB_PATH}")
print(f"Validation run ID: {RUN_ID}")
display(con.sql("SELECT version() AS duckdb_version").df())

## 5. Select tables to validate

Confirm which layers are in scope and optionally bootstrap the example pipeline from public TPC-H Parquet.

Set `BOOTSTRAP_EXAMPLE_PIPELINE = False` when tables already exist from `notebooks/01_etl_base.ipynb` or your own ETL.

In [ ]:
def table_exists(table_name: str) -> bool:
    schema, name = table_name.split(".", 1)
    result = con.sql(
        """
        SELECT COUNT(*) AS n
        FROM information_schema.tables
        WHERE table_schema = ? AND table_name = ?
        """,
        params=[schema, name],
    ).fetchone()[0]
    return result > 0


VALIDATION_TABLES = {
    "raw": RAW_TABLE_NAME,
    "staging": STAGING_TABLE_NAME,
    "curated": CURATED_TABLE_NAME,
}

print("Tables in scope:")
for layer, table in VALIDATION_TABLES.items():
    exists = table_exists(table)
    status = "exists" if exists else "missing"
    print(f"  {layer:8} → {table} ({status})")

all_exist = all(table_exists(t) for t in VALIDATION_TABLES.values())

if BOOTSTRAP_EXAMPLE_PIPELINE and not all_exist:
    con.execute("INSTALL httpfs; LOAD httpfs;")
    limit_clause = f"LIMIT {RAW_ROW_LIMIT}" if RAW_ROW_LIMIT else ""

    con.execute(f"""
    CREATE OR REPLACE TABLE {RAW_TABLE_NAME} AS
    SELECT * FROM read_parquet('{RAW_INPUT_PATH}')
    {limit_clause};
    """)

    con.execute(f"""
    CREATE OR REPLACE TABLE {STAGING_TABLE_NAME} AS
    SELECT
      l_orderkey::BIGINT AS order_id,
      l_linenumber::INTEGER AS line_number,
      l_suppkey::BIGINT AS supplier_id,
      CAST(l_shipdate AS DATE) AS order_date,
      TRY_CAST(l_extendedprice AS DOUBLE) AS amount,
      TRY_CAST(l_quantity AS DOUBLE) AS quantity,
      CASE
        WHEN l_returnflag = 'R' THEN 'returned'
        WHEN l_linestatus = 'F' THEN 'fulfilled'
        ELSE 'open'
      END AS order_status,
      l_shipmode AS ship_mode
    FROM {RAW_TABLE_NAME}
    WHERE l_orderkey IS NOT NULL;
    """)

    con.execute(f"""
    CREATE OR REPLACE TABLE {CURATED_TABLE_NAME} AS
    SELECT
      order_id,
      line_number,
      supplier_id,
      order_date,
      amount,
      quantity,
      order_status,
      ship_mode,
      DATE_TRUNC('month', order_date) AS order_month
    FROM {STAGING_TABLE_NAME}
    WHERE order_status <> 'returned'
      AND amount IS NOT NULL
      AND amount > 0;
    """)

    con.execute(f"""
    CREATE OR REPLACE TABLE curated.dim_suppliers AS
    SELECT DISTINCT supplier_id FROM {STAGING_TABLE_NAME} WHERE supplier_id IS NOT NULL;
    """)

    print("Bootstrapped example pipeline from TPC-H lineitem Parquet.")
elif not all_exist:
    missing = [t for t in VALIDATION_TABLES.values() if not table_exists(t)]
    raise RuntimeError(
        f"Missing tables: {missing}. Run 01_etl_base.ipynb or set BOOTSTRAP_EXAMPLE_PIPELINE = True."
    )
else:
    print("Using existing tables.")

layer_counts_sql = "\nUNION ALL\n".join(
    f"SELECT '{layer}' AS layer, '{table}' AS table_name, COUNT(*) AS row_count FROM {table}"
    for layer, table in VALIDATION_TABLES.items()
)
display(con.sql(layer_counts_sql).df())

## 6. Row count reconciliation

Compare row counts across layers. When `EXPECT_MONOTONIC_ROW_COUNTS` is true, a downstream layer should not exceed its upstream layer after documented filters.

**Pass:** violation query returns **zero rows**.

In [ ]:
row_count_violations_sql = f"""
WITH counts AS (
  SELECT 'raw' AS layer, COUNT(*) AS row_count FROM {RAW_TABLE_NAME}
  UNION ALL SELECT 'staging', COUNT(*) FROM {STAGING_TABLE_NAME}
  UNION ALL SELECT 'curated', COUNT(*) FROM {CURATED_TABLE_NAME}
),
pairs AS (
  SELECT
    r.row_count AS raw_count,
    s.row_count AS staging_count,
    c.row_count AS curated_count
  FROM counts r, counts s, counts c
  WHERE r.layer = 'raw' AND s.layer = 'staging' AND c.layer = 'curated'
)
SELECT 'staging_exceeds_raw' AS violation_type, staging_count AS count_a, raw_count AS count_b
FROM pairs WHERE staging_count > raw_count
UNION ALL
SELECT 'curated_exceeds_staging', curated_count, staging_count
FROM pairs WHERE curated_count > staging_count;
"""

row_count_summary_sql = f"""
SELECT '{RAW_TABLE_NAME}' AS layer, COUNT(*) AS row_count FROM {RAW_TABLE_NAME}
UNION ALL SELECT '{STAGING_TABLE_NAME}', COUNT(*) FROM {STAGING_TABLE_NAME}
UNION ALL SELECT '{CURATED_TABLE_NAME}', COUNT(*) FROM {CURATED_TABLE_NAME};
"""

print("--- Layer row counts (informational) ---")
display(con.sql(row_count_summary_sql).df())

if EXPECT_MONOTONIC_ROW_COUNTS:
    violations = con.sql(row_count_violations_sql).df()
    print("--- Row count violations (0 rows = pass) ---")
    if violations.empty:
        print("PASS: row counts are monotonic raw → staging → curated")
    else:
        display(violations)
    validation_results.append({
        "check_id": "VAL-001",
        "check_name": "row_count_reconciliation",
        "table_name": f"{RAW_TABLE_NAME} → {CURATED_TABLE_NAME}",
        "violating_rows": len(violations),
        "status": "PASS" if violations.empty else "FAIL",
        "detail": con.sql(row_count_summary_sql).df().to_string(index=False),
    })
else:
    print("EXPECT_MONOTONIC_ROW_COUNTS = False — skipping monotonic gate.")

## 7. Required field null checks

Return rows where any `REQUIRED_COLUMNS` value is NULL.

**Pass:** **zero rows** returned.

See `docs/09_validation/required_field_null_check.md`.

In [ ]:
if REQUIRED_COLUMNS:
    required_null_predicates = " OR ".join(f'"{col}" IS NULL' for col in REQUIRED_COLUMNS)
    required_null_violations_sql = f"""
    SELECT *
    FROM {PRIMARY_TABLE_NAME}
    WHERE {required_null_predicates}
    LIMIT {VIOLATION_SAMPLE_LIMIT};
    """
    required_null_count_sql = f"""
    SELECT COUNT(*) AS violating_rows
    FROM {PRIMARY_TABLE_NAME}
    WHERE {required_null_predicates};
    """

    null_violations = con.sql(required_null_violations_sql).df()
    null_count = con.sql(required_null_count_sql).fetchone()[0]

    print(f"--- Required-field null violations on {PRIMARY_TABLE_NAME} (0 rows = pass) ---")
    if null_violations.empty:
        print(f"PASS: no NULL values in {REQUIRED_COLUMNS}")
    else:
        display(null_violations)

    validation_results.append({
        "check_id": "VAL-003",
        "check_name": "required_field_null_check",
        "table_name": PRIMARY_TABLE_NAME,
        "violating_rows": null_count,
        "status": "PASS" if null_count == 0 else "FAIL",
        "detail": f"required: {', '.join(REQUIRED_COLUMNS)}",
    })
else:
    print("Set REQUIRED_COLUMNS to run required-field null checks.")

## 8. Primary key uniqueness checks

Return key combinations that appear more than once.

**Pass:** **zero rows** returned.

See `docs/09_validation/primary_key_uniqueness.md`.

In [ ]:
if PRIMARY_KEY_COLUMNS:
    pk_select = ", ".join(f'"{col}"' for col in PRIMARY_KEY_COLUMNS)
    pk_violations_sql = f"""
    SELECT
      {pk_select},
      COUNT(*) AS duplicate_count
    FROM {PRIMARY_TABLE_NAME}
    GROUP BY {pk_select}
    HAVING COUNT(*) > 1
    ORDER BY duplicate_count DESC
    LIMIT {VIOLATION_SAMPLE_LIMIT};
    """
    pk_count_sql = f"""
    SELECT COUNT(*) AS violating_groups FROM (
      SELECT {pk_select}
      FROM {PRIMARY_TABLE_NAME}
      GROUP BY {pk_select}
      HAVING COUNT(*) > 1
    );
    """

    pk_violations = con.sql(pk_violations_sql).df()
    pk_count = con.sql(pk_count_sql).fetchone()[0]

    print(f"--- Duplicate keys on {PRIMARY_TABLE_NAME} (0 rows = pass) ---")
    if pk_violations.empty:
        print(f"PASS: unique key for {PRIMARY_KEY_COLUMNS}")
    else:
        display(pk_violations)

    validation_results.append({
        "check_id": "VAL-002",
        "check_name": "primary_key_uniqueness",
        "table_name": PRIMARY_TABLE_NAME,
        "violating_rows": pk_count,
        "status": "PASS" if pk_count == 0 else "FAIL",
        "detail": f"key: {', '.join(PRIMARY_KEY_COLUMNS)}",
    })
else:
    print("Set PRIMARY_KEY_COLUMNS to run uniqueness checks.")

## 9. Referential integrity checks

Return orphan foreign-key rows with no matching parent record.

**Pass:** **zero rows** per FK check.

Configure `FOREIGN_KEY_CHECKS` in project configuration. See `docs/09_validation/referential_integrity.md`.

In [ ]:
if FOREIGN_KEY_CHECKS:
    for i, fk in enumerate(FOREIGN_KEY_CHECKS, start=1):
        fk_violations_sql = f"""
        SELECT
          c."{fk['child_key']}" AS orphan_key,
          COUNT(*) AS orphan_row_count
        FROM {fk['child_table']} c
        LEFT JOIN {fk['parent_table']} p
          ON c."{fk['child_key']}" = p."{fk['parent_key']}"
        WHERE p."{fk['parent_key']}" IS NULL
          AND c."{fk['child_key']}" IS NOT NULL
        GROUP BY c."{fk['child_key']}"
        ORDER BY orphan_row_count DESC
        LIMIT {VIOLATION_SAMPLE_LIMIT};
        """
        fk_count_sql = f"""
        SELECT COUNT(*) AS orphan_rows
        FROM {fk['child_table']} c
        LEFT JOIN {fk['parent_table']} p
          ON c."{fk['child_key']}" = p."{fk['parent_key']}"
        WHERE p."{fk['parent_key']}" IS NULL
          AND c."{fk['child_key']}" IS NOT NULL;
        """

        fk_violations = con.sql(fk_violations_sql).df()
        orphan_count = con.sql(fk_count_sql).fetchone()[0]
        check_label = fk.get("check_name", f"{fk['child_key']} → {fk['parent_table']}")

        print(f"--- FK: {check_label} (0 rows = pass) ---")
        if fk_violations.empty:
            print("PASS: no orphan foreign keys")
        else:
            display(fk_violations)

        validation_results.append({
            "check_id": f"VAL-004.{i:02d}",
            "check_name": "referential_integrity",
            "table_name": f"{fk['child_table']} → {fk['parent_table']}",
            "violating_rows": orphan_count,
            "status": "PASS" if orphan_count == 0 else "FAIL",
            "detail": check_label,
        })
else:
    print("Set FOREIGN_KEY_CHECKS to run referential integrity checks.")

## 10. Value range checks

Return rows where `NUMERIC_COLUMNS` fall outside `NUMERIC_BOUNDS`.

**Pass:** **zero rows** returned.

See `docs/09_validation/value_range_check.md`.

In [ ]:
if NUMERIC_COLUMNS:
    range_predicates = []
    for col in NUMERIC_COLUMNS:
        bounds = NUMERIC_BOUNDS.get(col, {})
        min_val = bounds.get("min")
        max_val = bounds.get("max")
        if min_val is not None:
            range_predicates.append(f'("{col}" IS NOT NULL AND "{col}" < {min_val})')
        if max_val is not None:
            range_predicates.append(f'("{col}" IS NOT NULL AND "{col}" > {max_val})')

    if range_predicates:
        range_where = " OR ".join(range_predicates)
        range_violations_sql = f"""
        SELECT *
        FROM {PRIMARY_TABLE_NAME}
        WHERE {range_where}
        LIMIT {VIOLATION_SAMPLE_LIMIT};
        """
        range_count_sql = f"""
        SELECT COUNT(*) AS violating_rows
        FROM {PRIMARY_TABLE_NAME}
        WHERE {range_where};
        """

        range_violations = con.sql(range_violations_sql).df()
        range_count = con.sql(range_count_sql).fetchone()[0]

        print(f"--- Value range violations on {PRIMARY_TABLE_NAME} (0 rows = pass) ---")
        if range_violations.empty:
            print(f"PASS: {NUMERIC_COLUMNS} within bounds")
        else:
            display(range_violations)

        validation_results.append({
            "check_id": "VAL-005",
            "check_name": "value_range_check",
            "table_name": PRIMARY_TABLE_NAME,
            "violating_rows": range_count,
            "status": "PASS" if range_count == 0 else "FAIL",
            "detail": f"columns: {', '.join(NUMERIC_COLUMNS)}",
        })
    else:
        print("Configure NUMERIC_BOUNDS for each column in NUMERIC_COLUMNS.")
else:
    print("Set NUMERIC_COLUMNS to run value range checks.")

## 11. Category domain checks

Return rows where categorical values are not in the allowed set defined in `CATEGORY_COLUMNS`.

**Pass:** **zero rows** per column.

See `docs/09_validation/category_domain_check.md`.

In [ ]:
if CATEGORY_COLUMNS:
    for i, (column, allowed_values) in enumerate(CATEGORY_COLUMNS.items(), start=1):
        if not allowed_values:
            continue
        allowed_list = ", ".join(f"'{v}'" for v in allowed_values)
        domain_violations_sql = f"""
        SELECT
          "{column}",
          COUNT(*) AS violating_rows
        FROM {PRIMARY_TABLE_NAME}
        WHERE "{column}" IS NOT NULL
          AND "{column}" NOT IN ({allowed_list})
        GROUP BY "{column}"
        ORDER BY violating_rows DESC
        LIMIT {VIOLATION_SAMPLE_LIMIT};
        """
        domain_count_sql = f"""
        SELECT COUNT(*) AS violating_rows
        FROM {PRIMARY_TABLE_NAME}
        WHERE "{column}" IS NOT NULL
          AND "{column}" NOT IN ({allowed_list});
        """

        domain_violations = con.sql(domain_violations_sql).df()
        domain_count = con.sql(domain_count_sql).fetchone()[0]

        print(f"--- Domain check: {column} (0 rows = pass) ---")
        if domain_violations.empty:
            print(f"PASS: all values in {allowed_values}")
        else:
            display(domain_violations)

        validation_results.append({
            "check_id": f"VAL-006.{i:02d}",
            "check_name": "category_domain_check",
            "table_name": f"{PRIMARY_TABLE_NAME}.{column}",
            "violating_rows": domain_count,
            "status": "PASS" if domain_count == 0 else "FAIL",
            "detail": f"domain: {', '.join(allowed_values)}",
        })
else:
    print("Set CATEGORY_COLUMNS to run domain checks.")

## 12. Date range checks

Return rows where `DATE_COLUMNS` fall outside `MIN_EXPECTED_DATE` / `MAX_EXPECTED_DATE`, or are in the future when `ALLOW_FUTURE_DATES = False`.

**Pass:** **zero rows** returned.

See `docs/09_validation/date_range_validation.md`.

In [ ]:
if DATE_COLUMNS:
    date_predicates = []
    for col in DATE_COLUMNS:
        date_predicates.append(
            f'("{col}" IS NOT NULL AND CAST("{col}" AS DATE) < DATE \'{MIN_EXPECTED_DATE}\')'
        )
        date_predicates.append(
            f'("{col}" IS NOT NULL AND CAST("{col}" AS DATE) > DATE \'{MAX_EXPECTED_DATE}\')'
        )
        if not ALLOW_FUTURE_DATES:
            date_predicates.append(
                f'("{col}" IS NOT NULL AND CAST("{col}" AS DATE) > CURRENT_DATE)'
            )

    date_where = " OR ".join(date_predicates)
    date_violations_sql = f"""
    SELECT *
    FROM {PRIMARY_TABLE_NAME}
    WHERE {date_where}
    LIMIT {VIOLATION_SAMPLE_LIMIT};
    """
    date_count_sql = f"""
    SELECT COUNT(*) AS violating_rows
    FROM {PRIMARY_TABLE_NAME}
    WHERE {date_where};
    """

    date_violations = con.sql(date_violations_sql).df()
    date_count = con.sql(date_count_sql).fetchone()[0]

    print(f"--- Date range violations on {PRIMARY_TABLE_NAME} (0 rows = pass) ---")
    if date_violations.empty:
        print(f"PASS: dates within [{MIN_EXPECTED_DATE}, {MAX_EXPECTED_DATE}]")
    else:
        display(date_violations)

    validation_results.append({
        "check_id": "VAL-007",
        "check_name": "date_range_validation",
        "table_name": PRIMARY_TABLE_NAME,
        "violating_rows": date_count,
        "status": "PASS" if date_count == 0 else "FAIL",
        "detail": f"columns: {', '.join(DATE_COLUMNS)}; window: {MIN_EXPECTED_DATE} to {MAX_EXPECTED_DATE}",
    })
else:
    print("Set DATE_COLUMNS to run date range checks.")

## 13. Aggregate reconciliation

Compare a numeric measure (`AGGREGATE_COLUMN`) between staging (with eligibility filter) and curated. **Pass** when the absolute delta is within `AGGREGATE_TOLERANCE`.

See `docs/09_validation/aggregate_reconciliation.md`.

In [ ]:
aggregate_reconciliation_sql = f"""
WITH staging_agg AS (
  SELECT
    COALESCE(SUM("{AGGREGATE_COLUMN}"), 0) AS total_amount,
    COUNT(*) AS row_count
  FROM {STAGING_TABLE_NAME}
  WHERE {STAGING_ELIGIBLE_FILTER}
),
curated_agg AS (
  SELECT
    COALESCE(SUM("{AGGREGATE_COLUMN}"), 0) AS total_amount,
    COUNT(*) AS row_count
  FROM {CURATED_TABLE_NAME}
)
SELECT
  'staging (eligible)' AS layer,
  s.total_amount,
  s.row_count
FROM staging_agg s
UNION ALL
SELECT 'curated', c.total_amount, c.row_count FROM curated_agg c
UNION ALL
SELECT
  'delta',
  c.total_amount - s.total_amount,
  c.row_count - s.row_count
FROM staging_agg s, curated_agg c;
"""

agg_df = con.sql(aggregate_reconciliation_sql).df()
delta_row = agg_df[agg_df["layer"] == "delta"].iloc[0]
amount_delta = abs(float(delta_row["total_amount"]))
row_delta = abs(int(delta_row["row_count"]))
agg_pass = amount_delta <= AGGREGATE_TOLERANCE and row_delta == 0

print(f"--- Aggregate reconciliation: {AGGREGATE_COLUMN} (informational + gate) ---")
display(agg_df)

validation_results.append({
    "check_id": "VAL-008",
    "check_name": "aggregate_reconciliation",
    "table_name": f"{STAGING_TABLE_NAME} → {CURATED_TABLE_NAME}",
    "violating_rows": 0 if agg_pass else 1,
    "status": "PASS" if agg_pass else "FAIL",
    "detail": f"amount_delta={amount_delta:.4f}, row_delta={row_delta}, tolerance={AGGREGATE_TOLERANCE}",
})

## 14. Spatial validation checks

When `GEOMETRY_COLUMN` is set, return features with null, empty, or invalid geometries.

**Pass:** **zero rows** returned.

Set `GEOMETRY_COLUMN = None` to skip. See `docs/09_validation/spatial_validity_check.md`.

In [ ]:
if RUN_SPATIAL_CHECKS and GEOMETRY_COLUMN:
    con.execute("INSTALL spatial; LOAD spatial;")
    geom = f'"{GEOMETRY_COLUMN}"'

    spatial_violations_sql = f"""
    SELECT
      *,
      CASE
        WHEN {geom} IS NULL THEN 'null geometry'
        WHEN ST_IsEmpty({geom}) THEN 'empty geometry'
        WHEN NOT ST_IsValid({geom}) THEN 'invalid geometry'
      END AS violation_type,
      ST_IsValidReason({geom}) AS invalid_reason
    FROM {SPATIAL_TABLE_NAME}
    WHERE {geom} IS NULL
       OR ST_IsEmpty({geom})
       OR NOT ST_IsValid({geom})
    LIMIT {VIOLATION_SAMPLE_LIMIT};
    """
    spatial_count_sql = f"""
    SELECT COUNT(*) AS violating_rows
    FROM {SPATIAL_TABLE_NAME}
    WHERE {geom} IS NULL
       OR ST_IsEmpty({geom})
       OR NOT ST_IsValid({geom});
    """

    spatial_violations = con.sql(spatial_violations_sql).df()
    spatial_count = con.sql(spatial_count_sql).fetchone()[0]

    print(f"--- Spatial validity on {SPATIAL_TABLE_NAME}.{GEOMETRY_COLUMN} (0 rows = pass) ---")
    if spatial_violations.empty:
        print("PASS: all geometries are present and valid")
    else:
        display(spatial_violations)

    validation_results.append({
        "check_id": "VAL-009",
        "check_name": "spatial_validity_check",
        "table_name": f"{SPATIAL_TABLE_NAME}.{GEOMETRY_COLUMN}",
        "violating_rows": spatial_count,
        "status": "PASS" if spatial_count == 0 else "FAIL",
        "detail": "null / empty / invalid geometry gate",
    })
else:
    print(
        "Skipping spatial checks. Set GEOMETRY_COLUMN (e.g. 'geom') and "
        "SPATIAL_TABLE_NAME to validate spatial layers."
    )

## 15. Validation summary table

Collect all check results into one DataFrame. **Overall pass** when every row has `status = 'PASS'`.

See `docs/09_validation/validation_summary_table.md`.

In [ ]:
summary_df = pd.DataFrame(validation_results)
summary_df.insert(0, "run_id", RUN_ID)
summary_df["run_ts"] = datetime.now(timezone.utc)

passed = (summary_df["status"] == "PASS").sum()
total = len(summary_df)
overall_pass = passed == total and total > 0

print(f"Validation result: {passed}/{total} checks passed")
display(summary_df[["check_id", "check_name", "table_name", "status", "violating_rows", "detail"]])

if overall_pass:
    print("\nOVERALL: PASS")
else:
    print("\nOVERALL: FAIL — review failing checks above")
    failed = summary_df[summary_df["status"] == "FAIL"]
    display(failed)

## 16. Export validation results

Persist the summary to `data/output/validation/` as Parquet and optionally register in `curated.validation_results`.

In [ ]:
if EXPORT_VALIDATION_RESULTS and not summary_df.empty:
    output_parquet = (VALIDATION_OUTPUT_DIR / f"{RUN_ID}_summary.parquet").resolve().as_posix()

    con.execute("""
    CREATE TABLE IF NOT EXISTS curated.validation_results (
      run_id VARCHAR,
      run_ts TIMESTAMP,
      check_id VARCHAR,
      check_name VARCHAR,
      table_name VARCHAR,
      status VARCHAR,
      violating_rows BIGINT,
      detail VARCHAR
    );
    """)

    con.register("validation_summary_df", summary_df)
    con.execute("INSERT INTO curated.validation_results SELECT * FROM validation_summary_df;")
    con.unregister("validation_summary_df")

    con.execute(f"""
    COPY (
      SELECT * FROM curated.validation_results WHERE run_id = '{RUN_ID}'
    ) TO '{output_parquet}' (FORMAT PARQUET, COMPRESSION ZSTD);
    """)

    print(f"Exported validation summary → {output_parquet}")
    print(f"Inserted {len(summary_df)} rows into curated.validation_results")
else:
    print("Skipping export (EXPORT_VALIDATION_RESULTS = False or no checks recorded).")

## 17. Final validation notes

Document decisions, waivers, and next steps for this validation run.

### Assumptions (example — replace for your project)

- **Source:** TPC-H `lineitem` Parquet from `shell.duckdb.org` stands in for an orders feed; swap table names and column lists for your pipeline.
- **Grain:** Composite key `order_id` + `line_number` at staging; curated excludes returned lines and non-positive amounts.
- **FK example:** `supplier_id` validated against `curated.dim_suppliers` built from distinct staging values.
- **Spatial:** Not used in this example; set `GEOMETRY_COLUMN = 'geom'` after spatial ingest (`docs/03_spatial_ingestion/`).
- **Tolerance:** `AGGREGATE_TOLERANCE = 0.01` on `amount` sum — tighten for financial reconciliation.

### Next actions

- [ ] **Export** — write curated table to `output` (`notebooks/01_etl_base.ipynb` section 12, `docs/10_export/`)
- [ ] **Spatial EDA** — profile geometry before delivery (`notebooks/02_spatial_eda_base.ipynb`)
- [ ] **Cleaning** — fix violations surfaced above (`docs/06_cleaning/`)
- [ ] **CI gate** — assert `overall_pass` in automation (`docs/09_validation/validation_summary_table.md`)
- [ ] **Productionize** — move check SQL to `sql/validation/` scripts

---

_Close the DuckDB connection when finished, or leave it open for the next notebook in the session._

In [ ]:
# con.close()
# print("Connection closed.")